In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:2]

['emma', 'olivia']

In [3]:
len(words)

32033

In [4]:
#Build the vocabulary of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [5]:
# Building The dataset

block_size = 3 # context length : how many characters we take to predict the next one

X, Y = [], []
for w in words[:1]:
    print(w)
    
    # context = [0]
    # print(context)
    
    context = [0] * block_size
    # print(context)

    # x = w + '.'
    
    # print(x)
    
    for ch in w + '.':
        ix = stoi[ch] # 5
        X.append(context)
        # print(X)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '------>', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)
        

emma
... ------> e
..e ------> m
.em ------> m
emm ------> a
mma ------> .


In [6]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])

In [7]:
Y

tensor([ 5, 13, 13,  1,  0])

In [8]:
seed = torch.Generator().manual_seed(123456)

In [9]:
# Embedding Table / Glorified Lookup table
C = torch.randn((27,2), generator=seed)
C

tensor([[-1.1218, -1.8445],
        [ 1.0258,  2.1921],
        [-1.4322,  1.5293],
        [ 1.2118,  0.0423],
        [ 0.3807,  0.0538],
        [ 0.5047, -0.4987],
        [-0.4891,  0.0578],
        [ 0.9436,  0.9775],
        [ 0.7843, -0.1034],
        [ 0.1187, -0.2737],
        [ 1.1537,  1.3910],
        [ 1.0732, -1.2210],
        [ 0.9103,  0.8257],
        [ 0.1247,  0.8079],
        [ 0.1415, -0.5531],
        [ 1.0828, -1.7865],
        [-0.7372, -0.7429],
        [-0.4891,  2.4613],
        [-0.7179, -0.2446],
        [-0.8818,  1.6565],
        [-0.5419, -2.4956],
        [ 0.3731,  1.2806],
        [-0.1539,  0.2954],
        [-1.5935, -0.0451],
        [-0.2014, -0.7450],
        [-0.2250,  0.0894],
        [-1.0029, -1.1601]])

In [10]:
C[5]

tensor([ 0.5047, -0.4987])

In [11]:
emb = C[X]
emb.shape

torch.Size([5, 3, 2])

In [12]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [13]:
emb

tensor([[[-1.1218, -1.8445],
         [-1.1218, -1.8445],
         [-1.1218, -1.8445]],

        [[-1.1218, -1.8445],
         [-1.1218, -1.8445],
         [ 0.5047, -0.4987]],

        [[-1.1218, -1.8445],
         [ 0.5047, -0.4987],
         [ 0.1247,  0.8079]],

        [[ 0.5047, -0.4987],
         [ 0.1247,  0.8079],
         [ 0.1247,  0.8079]],

        [[ 0.1247,  0.8079],
         [ 0.1247,  0.8079],
         [ 1.0258,  2.1921]]])

In [14]:
torch.cat((emb[:,0,:],emb[:,1,:],emb[:,2,:]),dim=1)

tensor([[-1.1218, -1.8445, -1.1218, -1.8445, -1.1218, -1.8445],
        [-1.1218, -1.8445, -1.1218, -1.8445,  0.5047, -0.4987],
        [-1.1218, -1.8445,  0.5047, -0.4987,  0.1247,  0.8079],
        [ 0.5047, -0.4987,  0.1247,  0.8079,  0.1247,  0.8079],
        [ 0.1247,  0.8079,  0.1247,  0.8079,  1.0258,  2.1921]])

In [15]:
torch.cat(torch.unbind(emb,1),1)

tensor([[-1.1218, -1.8445, -1.1218, -1.8445, -1.1218, -1.8445],
        [-1.1218, -1.8445, -1.1218, -1.8445,  0.5047, -0.4987],
        [-1.1218, -1.8445,  0.5047, -0.4987,  0.1247,  0.8079],
        [ 0.5047, -0.4987,  0.1247,  0.8079,  0.1247,  0.8079],
        [ 0.1247,  0.8079,  0.1247,  0.8079,  1.0258,  2.1921]])

In [16]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [17]:
a.shape

torch.Size([18])

In [18]:
a.view(3,3,2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [19]:
a.storage()

/tmp/ipykernel_526714/214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [20]:
emb.storage()

 -1.1218369007110596
 -1.8445433378219604
 -1.1218369007110596
 -1.8445433378219604
 -1.1218369007110596
 -1.8445433378219604
 -1.1218369007110596
 -1.8445433378219604
 -1.1218369007110596
 -1.8445433378219604
 0.504667341709137
 -0.49874359369277954
 -1.1218369007110596
 -1.8445433378219604
 0.504667341709137
 -0.49874359369277954
 0.12470921874046326
 0.8078563213348389
 0.504667341709137
 -0.49874359369277954
 0.12470921874046326
 0.8078563213348389
 0.12470921874046326
 0.8078563213348389
 0.12470921874046326
 0.8078563213348389
 0.12470921874046326
 0.8078563213348389
 1.02579927444458
 2.192124843597412
[torch.storage.TypedStorage(dtype=torch.float32, device=cpu) of size 30]

In [21]:
emb.view(5,6)

tensor([[-1.1218, -1.8445, -1.1218, -1.8445, -1.1218, -1.8445],
        [-1.1218, -1.8445, -1.1218, -1.8445,  0.5047, -0.4987],
        [-1.1218, -1.8445,  0.5047, -0.4987,  0.1247,  0.8079],
        [ 0.5047, -0.4987,  0.1247,  0.8079,  0.1247,  0.8079],
        [ 0.1247,  0.8079,  0.1247,  0.8079,  1.0258,  2.1921]])

In [22]:
h = emb.view(-1,6) @ W1 + b1

In [23]:
h

tensor([[ 1.0923e+00,  2.0868e+00,  3.1194e+00, -4.8435e+00, -6.6008e+00,
         -7.1583e-01, -4.4284e+00, -4.2549e+00,  1.5107e+00, -3.9028e+00,
         -5.8145e+00,  3.9181e+00, -2.2178e+00, -2.0669e-01, -6.7423e+00,
          6.5405e-01, -4.7135e+00, -4.1581e+00,  4.7429e+00, -1.3580e+00,
          2.8412e+00, -5.3123e-01, -3.7167e+00,  4.5327e+00,  5.6356e+00,
          2.6299e+00, -3.3194e+00,  1.9374e+00,  4.5313e-01, -2.3218e-01,
         -2.4658e+00,  3.7229e+00,  1.8070e+00,  2.3454e+00, -6.1194e+00,
          9.9372e+00,  6.1554e-01, -3.3569e+00,  7.9443e+00,  9.9182e+00,
          3.0262e+00, -5.1836e+00,  1.4423e+00, -4.7232e+00, -1.8562e+00,
         -5.6391e+00, -1.1299e+00, -8.9091e+00,  5.3935e+00, -1.6188e+00,
         -1.8153e+00, -8.4180e+00, -1.6089e+00, -1.1561e-01, -3.4224e+00,
          6.8547e-01,  7.8040e-01,  8.8296e+00, -6.4152e+00,  1.4761e+00,
         -1.6473e+00, -2.5989e+00, -2.6607e+00, -2.2970e+00,  1.9589e+00,
          8.7603e-01,  2.3109e+00, -8.

In [24]:
h = torch.tanh(h)

In [25]:
h.shape

torch.Size([5, 100])

In [26]:
W2 = torch.randn((100,27), generator=seed)
b2 = torch.randn(27, generator=seed)

In [27]:
logits = h @ W2 + b2
logits

tensor([[-13.5127,  14.8071,  -6.0347,  -7.1791,   4.4916, -15.2951,  16.7352,
         -21.3859,  -1.7188,   4.4440,  -0.8781,   3.4491,  15.5617,   5.6854,
          -9.8885,  19.8687,   0.9406,   1.7211, -16.0492,   2.1712,  -4.7719,
          11.3173,  -2.3133,  12.3469,  -9.3400,  -2.0264,  -8.3419],
        [-10.6578,  10.1303,  -4.0656,  -4.4525,   1.3079, -22.4800,  14.3120,
         -15.3028,  -6.0978,  -0.1737,   4.0258,   3.1664,  18.1715,   4.2924,
          -3.1196,  16.8761,   2.0223,  -0.1599,  -9.7978,   0.6421, -11.5024,
          10.4068,   2.6796,   3.9357,   5.4157,   7.1594,  -6.2125],
        [ -6.5497,   3.1407, -13.5279,  -1.1742,   3.2690,  -3.3688,  -0.4553,
         -11.9915,  -0.5300,   4.3339,   5.8799,  -3.2217,  18.6079,  14.8781,
          -6.7885,  -3.7308,   1.5727,   7.5434, -12.0341,  -2.3526,   7.0170,
          -3.0335,  -4.4764,  -6.5866,  -3.5341,   1.9033,   3.9948],
        [ 11.1499,  -3.6242,  -2.1917,   1.2073,  -9.6556,  -5.0791, -12.8740,


In [28]:
logits.shape

torch.Size([5, 27])

In [29]:
counts = logits.exp()

In [30]:
probs = counts / counts.sum(1, keepdims=True)
probs

tensor([[2.9899e-15, 5.9538e-03, 5.2882e-12, 1.6840e-12, 1.9717e-07, 5.0300e-16,
         4.0943e-02, 1.1386e-18, 3.9601e-10, 1.8801e-07, 9.1796e-10, 6.9517e-08,
         1.2663e-02, 6.5060e-07, 1.1211e-13, 9.3975e-01, 5.6579e-09, 1.2349e-08,
         2.3663e-16, 1.9369e-08, 1.8696e-11, 1.8163e-04, 2.1853e-10, 5.0859e-04,
         1.9402e-13, 2.9113e-10, 5.2641e-13],
        [2.3288e-13, 2.4847e-04, 1.6985e-10, 1.1536e-10, 3.6625e-08, 1.7092e-18,
         1.6270e-02, 2.2378e-15, 2.2259e-11, 8.3238e-09, 5.5479e-07, 2.3490e-07,
         7.7183e-01, 7.2427e-07, 4.3743e-10, 2.1131e-01, 7.4822e-08, 8.4389e-09,
         5.5030e-13, 1.8820e-08, 1.0008e-13, 3.2761e-04, 1.4437e-07, 5.0698e-07,
         2.2272e-06, 1.2735e-05, 1.9846e-11],
        [1.1584e-11, 1.8723e-07, 1.0796e-14, 2.5027e-09, 2.1284e-07, 2.7882e-10,
         5.1361e-09, 5.0178e-14, 4.7662e-09, 6.1742e-07, 2.8973e-06, 3.2301e-10,
         9.7654e-01, 2.3433e-02, 9.1237e-12, 1.9414e-10, 3.9027e-08, 1.5290e-05,
         4.8088e-

In [31]:
probs.shape

torch.Size([5, 27])

In [32]:
probs[0].sum()

tensor(1.)

In [33]:
loss = -probs[torch.arange(5),Y].log().mean()
loss

tensor(14.9977)

In [34]:
# Model

In [35]:
X.shape, Y.shape

(torch.Size([5, 3]), torch.Size([5]))

In [36]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2),generator=g, requires_grad=True)
W1 = torch.randn((6,100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100,27), generator=g, requires_grad=True) * 0.1
b2 = torch.randn(27,generator=g, requires_grad=True) * 0.0
parameters = [C, W1, b1, W2, b2]

In [37]:
sum(p.nelement() for p in parameters)

3481

In [38]:
for _ in range(10):
    # Forward Pass
    xemb = C[X] # 5,3,2
    # --- Neuron ----
    h = xemb.view(-1,6) @ W1 + b1
    h = torch.tanh(h)
    
    # --- Loss Calc ----
    logits = h @ W2 + b2
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # loss = -probs[torch.arange(5),Y].log().mean()
    loss = F.cross_entropy(logits,Y)
    print(loss.item())
    
    # Back Prop
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # Update
    for p in parameters:
        p.data += -0.01 * p.grad

4.078691005706787


/tmp/ipykernel_526714/3427339421.py:23: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  p.data += -0.01 * p.grad


TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [ ]:
logits.max(1)

# MLP Explained

In [ ]:
# === Building a dataset ===
xs = []
ys = []

block_size = 3
content = [0] * 3

for w in words[:1]: 
    for ch in w + '.':
        ix = stoi[ch]
        xs.append(content)
        ys.append(ix)
        content = content[1:] + [ix]

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs,ys

In [ ]:
seedz = torch.Generator().manual_seed(2147483647) # For making it deterministic

In [ ]:
# === LAYER BUILDING ===
Cz = torch.randn((27,2),generator=seedz, requires_grad=True) # embedding Matrix

In [ ]:
# ==== Embedding + Foward Pass ===
xs_embedded = Cz[xs]

In [ ]:
# === 1st layer ===
# XS contains a shape of [...,3(context window),2(embedding vectors)]
W1 = torch.randn((6,100), generator=seedz, requires_grad=True) # weights matrix
b1 = torch.randn((100), generator=seedz, requires_grad=True) # bias matrix

In [ ]:
# === Defining Neurons in 1st Layer + Forward Pass ===
h = xs_embedded.view(-1,6) @ W1 + b1
h = torch.tanh(h)

In [ ]:
# === 2nd Layer ===

W2 = torch.randn((100,27), generator=seedz, requires_grad=True)
b2 = torch.randn((27), generator=seedz, requires_grad=True)

In [ ]:
parameters = [Cz, W1, b1, W2, b2]

In [ ]:
# === Defining Neurons in 2nd Layer + Forward Pass ===
logits = h @ W2 + b2

In [ ]:
# === Loss Calculation
loss = F.cross_entropy(logits, ys)
print(loss.item())

In [ ]:
# === Backpropagation ===
for p in parameters:
    p.grad = None
    
loss.backward()

In [ ]:
# === Updating Data
for p in parameters:
    p.data += -0.1 * p.grad

# GPU Bigram MLP

In [ ]:
device = "cuda" if torch.cuda.is_available else "cpu"
device

In [ ]:
def build_dataset(words):
    block_size = 3
    gx = []
    gy = []
    
    for w in words:
        content = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            gx.append(content)
            gy.append(ix)
            content = content[1:] + [ix]

    gx = torch.tensor(gx).to(device)
    gy = torch.tensor(gy).to(device)
    print(gx.shape,gy.shape)
    return gx,gy

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev= build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [ ]:
Xtr.shape, Ytr.shape

In [ ]:
g = torch.Generator().manual_seed(2147483647)

In [ ]:
gC = torch.randn((27,2), generator=g).to(device)
gW1 = torch.randn((51,579), generator=g).to(device) * (5/3) / (51**0.5)
gb1 = torch.randn((579), generator=g).to(device) * 0
gW2 = torch.randn((579,27), generator=g).to(device) * 0.01
gb2 = torch.randn((27), generator=g).to(device) * 0

gbngain = torch.ones((1,579)).to(device)
gbnbias = torch.zeros((1,579)).to(device)

parameters = [gC, gW1, gb1, gW2, gb2, gbngain, gbnbias]

In [ ]:
sum(p.nelement() for p in parameters)

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
lre = torch.linspace(-3,0,1000)
lrs = 10**lre

In [ ]:
lri = []
lossi = []
stepi = []

In [ ]:
for i in range(1000):

    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (43,))
    
    # Forward Pass
    gx_emb = gC[Xtr[ix]]
    preact = gx_emb.view(-1,51) @ gW1 + gb1
    bnmean = preact.mean(0, keepdim=True)
    bnstd = preact.std(0, keepdim=True)
    preact = preact - bnmean / bnstd
    h = torch.tanh(preact)
    logits = h @ gW2 + gb2
    loss = F.cross_entropy(logits,Ytr[ix])
    #print(loss.item())
    
    # Back Propagation
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # Update
    lr = 0.1 #if i < 50000 else 0.001
    for p in parameters:
        p.data += -lr * p.grad

    #track stats
    # lri.append(lrs[i])
    stepi.append(i)
    lossi.append(loss.log10().item())
#print(loss.item())

In [ ]:
print(loss.item())

In [ ]:
h.shape

In [ ]:
plt.figure(figsize=(20,10))
plt.imshow(torch.Tensor.cpu(h).abs() > 0.99, cmap='gray', interpolation="nearest");

In [ ]:
plt.hist(torch.Tensor.cpu(h).view(-1).tolist(),50);

In [ ]:
plt.hist(torch.Tensor.cpu(preact).view(-1).tolist(),50);

In [ ]:
-torch.tensor(1/27.0).log()

In [ ]:
x = torch.randn(1000,10)
w = torch.randn(10,200) / 10**0.5 # gain / square root of fan_n ~ (5/3) / ([10 as input neurons]**0.5) to keep the weights in unit deviation.
logi = x @ w
print(x.mean(),x.std())
print(w.mean(),w.std())
print(logi.mean(),logi.std())
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.hist(x.view(-1).tolist(),50);
plt.subplot(122)
plt.hist(logi.view(-1).tolist(),50);

In [ ]:
plt.plot(stepi,lossi)

In [ ]:
preact.mean(0, keepdim=True).shape

In [ ]:
gx_emb = gC[Xtr]
h = torch.tanh(gx_emb.view(-1,51) @ gW1 + gb1)
logits = h @ gW2 + gb2
loss = F.cross_entropy(logits,Ytr)
print(loss.item())

In [ ]:
gx_emb = gC[Xdev]
h = torch.tanh(gx_emb.view(-1,51) @ gW1 + gb1)
logits = h @ gW2 + gb2
loss = F.cross_entropy(logits,Ydev)
print(loss.item())

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(gC[:,0].to(device='cpu').data,gC[:,1].to(device='cpu').data, s=200)
for i in range(gC.shape[0]):
    plt.text(gC[i,0].to(device='cpu').item(),gC[i,1].to(device='cpu').item(),itos[i],ha="center",va="center",color="white")
plt.grid('minor')

In [ ]:
# training split, dev/validation split, test split.
# 80% , 10%, 10%

In [ ]:
gx_emb = gC[Xte]
h = torch.tanh(gx_emb.view(-1,51) @ gW1 + gb1)
logits = h @ gW2 + gb2
loss = F.cross_entropy(logits,Yte)
print(loss.item())

# Sampling

In [ ]:
for _ in range(20):

    out = []
    context = [0] * block_size

    while True:
        gemb = gC[torch.tensor([context])]
        h = torch.tanh(gemb.view(1,-1) @ gW1 + gb1)
        logits = h @ gW2 + gb2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1,generator=gg).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix ==0:
            break

    print(''.join(itos[i] for i in out))